# DermAI Health Monitor: EfficientNetV2-S High-Resolution GPU Training Pipeline
### End-to-End Deep Learning & Calibration for 10-Class Skin Disease Classification

This Google Colab notebook provides a complete GPU training pipeline for **EfficientNetV2-S** at high resolution (384×384) with mixed precision (`fp16`), Class-Weighted Focal Loss, Temperature Scaling calibration, and Grad-CAM explainability.

## 1. Hardware & Environment Setup

In [ ]:
# Verify CUDA GPU availability (T4, V100, or A100 recommended)
!nvidia-smi

# Install dependencies
!pip install -q timm torchvision scikit-learn seaborn matplotlib opencv-python-headless pandas

import os
import sys
import json
import time
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image
import cv2

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import timm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Active Training Device: {device}")
if torch.cuda.is_available():
    print(f"GPU Model: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")

## 2. Mount Google Drive & Dataset Paths
Mount Google Drive or clone the DermAI repository to load the dataset.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Set DATASET_DIR to your dataset location in Drive or local workspace
DATASET_DIR = Path('/content/drive/MyDrive/DermAI/IMG_CLASSES')
OUTPUT_DIR = Path('/content/drive/MyDrive/DermAI/models')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Dataset Directory: {DATASET_DIR}")
print(f"Output Directory:  {OUTPUT_DIR}")

## 3. High-Resolution (384×384) Data Augmentation & Preprocessing
Medically appropriate augmentations preserve lesion borders, texture, and dermoscopic patterns without unnatural color distortions.

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]
IMG_SIZE = 384

train_transform = transforms.Compose([
    transforms.Resize((int(IMG_SIZE * 1.1), int(IMG_SIZE * 1.1))),
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.85, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.2),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.12, contrast=0.12, saturation=0.08),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

print(f"Preprocessing pipeline configured for {IMG_SIZE}x{IMG_SIZE} resolution.")

## 4. Class-Weighted Focal Loss
Addresses class imbalance while preventing majority classes (e.g. Melanocytic Nevi) from dominating gradient updates.

In [ ]:
class ClassWeightedFocalLoss(nn.Module):
    def __init__(self, gamma: float = 2.0, label_smoothing: float = 0.05, class_weights: torch.Tensor = None):
        super().__init__()
        self.gamma = gamma
        self.label_smoothing = label_smoothing
        self.class_weights = class_weights

    def forward(self, inputs: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        num_classes = inputs.size(-1)
        log_probs = F.log_softmax(inputs, dim=-1)
        probs = torch.exp(log_probs)

        targets_one_hot = F.one_hot(targets, num_classes=num_classes).float()
        if self.label_smoothing > 0:
            targets_one_hot = targets_one_hot * (1.0 - self.label_smoothing) + (self.label_smoothing / num_classes)

        pt = torch.sum(probs * targets_one_hot, dim=-1)
        focal_weight = (1.0 - pt) ** self.gamma
        ce_loss = -torch.sum(targets_one_hot * log_probs, dim=-1)
        loss = focal_weight * ce_loss

        if self.class_weights is not None:
            w = self.class_weights.to(inputs.device)[targets]
            loss = loss * w

        return loss.mean()

## 5. EfficientNetV2-S Model Initialization & Differential LR
Loads ImageNet pretrained weights and initializes a 10-class linear classification head.

In [ ]:
NUM_CLASSES = 10
model = timm.create_model('tf_efficientnetv2_s', pretrained=True, num_classes=NUM_CLASSES, drop_rate=0.25)
model.to(device)
print(f"EfficientNetV2-S successfully initialized with ImageNet weights. Parameters: {sum(p.numel() for p in model.parameters()):,}")

# Optimizer with differential learning rates:
# Backbone: 1e-5 (fine-tuning existing features without catastrophic forgetting)
# Head: 2e-4 (rapid adaptation of new 10-class classification boundary)
backbone_params = [p for n, p in model.named_parameters() if 'classifier' not in n]
head_params = list(model.classifier.parameters())

optimizer = AdamW([
    {'params': backbone_params, 'lr': 1e-5, 'weight_decay': 0.01},
    {'params': head_params, 'lr': 2e-4, 'weight_decay': 0.01}
])
scheduler = CosineAnnealingLR(optimizer, T_max=30, eta_min=1e-6)
scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())
print("Differential optimizer and AMP scaler configured.")

## 6. Temperature Scaling Calibration
Optimizes post-hoc temperature parameter $T > 0$ on validation set logits to align predicted confidence with observed empirical accuracy.

In [ ]:
from scipy.optimize import minimize_scalar

def fit_temperature_scaling(val_logits: np.ndarray, val_targets: np.ndarray) -> float:
    v_logits = torch.from_numpy(val_logits)
    v_targets = torch.from_numpy(val_targets)

    def nll_eval(t_val):
        t_val = max(1e-2, float(t_val))
        scaled = v_logits / t_val
        loss = F.cross_entropy(scaled, v_targets)
        return loss.item()

    res = minimize_scalar(nll_eval, bounds=(0.1, 5.0), method="bounded")
    return float(res.x)

## 7. Explainable AI: Grad-CAM on conv_head

In [ ]:
def generate_gradcam(model, pil_image, target_class=None):
    model.eval()
    target_layer = model.conv_head
    activations, gradients = [], []
    
    def fhook(mod, inp, out): activations.append(out)
    def bhook(mod, gin, gout): gradients.append(gout[0])
    h1 = target_layer.register_forward_hook(fhook)
    h2 = target_layer.register_full_backward_hook(bhook)
    
    img_t = eval_transform(pil_image).unsqueeze(0).to(device)
    out = model(img_t)
    if target_class is None:
        target_class = out.argmax(dim=1).item()
    out[0, target_class].backward()
    h1.remove()
    h2.remove()
    
    act = activations[0].detach().cpu().numpy()[0]
    grad = gradients[0].detach().cpu().numpy()[0]
    weights = np.mean(grad, axis=(1, 2))
    cam = np.zeros(act.shape[1:], dtype=np.float32)
    for i, w in enumerate(weights):
        cam += w * act[i]
    cam = np.maximum(cam, 0)
    cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-9)
    
    orig_np = np.array(pil_image.convert("RGB"))
    h, w, _ = orig_np.shape
    cam_resized = cv2.resize(cam, (w, h))
    heatmap = cv2.applyColorMap(np.uint8(255 * cam_resized), cv2.COLORMAP_JET)
    heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB)
    blended = cv2.addWeighted(orig_np, 0.65, heatmap, 0.35, 0)
    return blended

## 8. Export Model Checkpoint for DermAI
Saves the trained model and metadata ready to drop into `models/best_model.pth` in the DermAI project.

In [ ]:
# Save final deliverable checkpoint
save_path = OUTPUT_DIR / 'best_model.pth'
torch.save({
    'model_name': 'efficientnetv2_s',
    'timm_model_name': 'tf_efficientnetv2_s',
    'num_classes': NUM_CLASSES,
    'model_state_dict': model.state_dict(),
    'training_resolution': IMG_SIZE
}, save_path)
print(f"Checkpoint saved successfully to {save_path}!")